# Tableau comparatif final — Tous les modèles

Tableau comparatif final de tous les modèles testés (Isolation Forest, One-Class SVM, LOF, Autoencoder, LSTM, GRU). Métriques : précision, recall, F1, AUC-ROC, temps d'inférence, consommation mémoire.

## Importation des librairies

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time
import tracemalloc  # mesure la consommation mémoire
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)


2026-05-01 10:13:39.506796: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
chemin_CICIDS = '../Data/cicids_clean.csv'
chemin_UNSW   = '../Data/unsw_clean.csv'
chemin_LOGS   = '../Data/logs_clean.csv'

df_cicids = pd.read_csv(chemin_CICIDS, low_memory=False)
df_unsw = pd.read_csv(chemin_UNSW,low_memory=False)
df_logs = pd.read_csv(chemin_LOGS,low_memory=False)


## Fonctions utilitaires

On crée plusieurs fonctions utilitaires qui nous aideront tout le long de ce notebook afin d'effectuer les comparaisons.

Cette 1ère fonction effectue la mesure du temps d'inférence le but est de savoir combien de temps le modèle met à faire des prédictions ainsi on répète n=5 fois pour avoir une mesure stable.

In [3]:
def mesurer_inference(model, X, n=5):
    temps = [] #liste vide
    for _ in range(n):
        t0 = time.time() 
        model.predict(X) if hasattr(model, 'predict') else model.fit_predict(X)
        temps.append(time.time() - t0) #temps final
    return round(np.mean(temps), 4)  # moyenne des n mesures en secondes

Cette 2ème fonction effectue la mesure de la consommation mémoire ainsi le but est de savoir combien de RAM le modèle consomme pendant l'inférence
On utilise tracemalloc qui est une librairie Python qui surveille les allocations mémoire

In [4]:
def mesurer_memoire(fn):
    tracemalloc.start()        # démarre la surveillance mémoire
    fn()                       # exécute la fonction à mesurer
    _, peak = tracemalloc.get_traced_memory()  # récupère le pic de mémoire en bytes
    tracemalloc.stop()
    return round(peak / 1024 / 1024, 2)  # conversion bytes en MB

Cette 3ème fonction effectue la construction de l'Autoencoder ayant pour but de créer le réseau de neurones encodeur/décodeur, on réutilise donc la meilleure architecture que dans le notebook autoencoder

In [5]:
def build_autoencoder(input_dim, latent_dim=8, hidden=[32]):
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    for u in hidden:                              # encodeur qui réduit la dimension
        x = layers.Dense(u, activation='relu')(x)
    x = layers.Dense(latent_dim, activation='relu')(x)  # espace latent
    for u in reversed(hidden):                    # décodeur qui reconstruit
        x = layers.Dense(u, activation='relu')(x)
    outputs = layers.Dense(input_dim, activation='linear')(x)  # sortie de même dimension que l'entrée
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model

La 4ème fonction réalise la construction du LSTM avecpour but de créer le réseau récurrent LSTM pour la prédiction de séquences

In [6]:
def build_lstm(seq_len, n_features, units=64):
    model = keras.Sequential([
        layers.LSTM(units, input_shape=(seq_len, n_features)),  # lit la séquence
        layers.Dense(n_features)                                 # prédit le point suivant
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


La 5ème effectue la construction du GRU afin de créer une version simplifiée du LSTM, plus rapide avec des performances similaires

In [7]:
def build_gru(seq_len, n_features, units=64):
    model = keras.Sequential([
        layers.GRU(units, input_shape=(seq_len, n_features)),
        layers.Dense(n_features)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

La 6ème fonction effectue la création des séquences temporelles avec pour but de transformer les données brutes en fenêtres glissantes pour LSTM/GRU comme ce que l'on avait fait précédemment dans le notebook lstm_gru.ipynb


In [8]:
def creer_sequences(X, seq_len=10):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i + seq_len])  # fenêtre de seq_len points
        y_seq.append(X[i + seq_len])    # point suivant à prédire
    return np.array(X_seq), np.array(y_seq)


Finalement la dernière fonction fait l'évaluation complète d'un modèle baseline afin d'entraîner et évaluer IF/LOF/SVM et retourner toutes les métriques en une fois
Il s'agit de la fonction principale qui orchestre mesurer_memoire et les métriques sklearn

In [9]:
def evaluer_baseline(model, X, y, nom, dataset):
    t0   = time.time() # temps d'inférence
    pred = (model.fit_predict(X) == -1).astype(int)  # -1 pour anomalie en 1 sinon  0
    t_inf = round(time.time() - t0, 3)

    # score continu pour l'AUC et chaque modèle a son propre attribut
    if hasattr(model, 'negative_outlier_factor_'):  # LOF
        score = -model.negative_outlier_factor_
    elif hasattr(model, 'score_samples'):            # Isolation Forest
        score = -model.score_samples(X)
    else:                                            # One-Class SVM
        score = -model.decision_function(X)

    mem = mesurer_memoire(lambda: model.fit_predict(X)) # mémoire consommée pendant fit_predict

    return {
        'Dataset'     : dataset,
        'Modèle'      : nom,
        'Précision'   : round(precision_score(y, pred, zero_division=0), 3),
        'Recall'      : round(recall_score(y, pred, zero_division=0), 3),
        'F1'          : round(f1_score(y, pred, zero_division=0), 3),
        'AUC-ROC'     : round(roc_auc_score(y, score), 3),
        'Temps (s)'   : t_inf,
        'Mémoire (MB)': mem}


## Dataset 1 : CIC-IDS-2017

In [10]:
# Préparation
X = df_cicids.drop(columns=['Label']).select_dtypes(include=[np.number])
y_cicids = (df_cicids['Label'] != 'BENIGN').astype(int)
X = X.fillna(0).replace([np.inf, -np.inf], 0)
X_cicids_scaled = StandardScaler().fit_transform(X)

# échantillon  de 10 000 points
X_c, _, y_c, _ = train_test_split(X_cicids_scaled, y_cicids, train_size=10000, stratify=y_cicids, random_state=42)

print(f"Échantillon du dataset CICIDS : {X_c.shape} et {y_c.mean():.2%} d'attaques")

Échantillon du dataset CICIDS : (10000, 79) et 57.38% d'attaques


In [11]:
resultats_cicids = []

# on utilise evaluer_baseline qui calcule toutes les métriques en une seule fois
for nom, modele in [
    ('Isolation Forest', IsolationForest(contamination=0.50, random_state=42, n_jobs=-1)),
    ('LOF',              LocalOutlierFactor(n_neighbors=5, contamination=0.50)),
    ('One-Class SVM',    OneClassSVM(kernel='rbf', nu=0.50, gamma='scale'))]:
    print(f"Évaluation {nom}...")
    resultats_cicids.append(evaluer_baseline(modele, X_c, y_c, nom, 'CICIDS2017'))


#  Autoencoder
# séparation normaux et attaques 
X_normal  = X_cicids_scaled[y_cicids == 0][:20000]  # échantillon de 20 000 normaux
X_attaque = X_cicids_scaled[y_cicids == 1]

# 80% train (normaux) et 20% test (normaux et attaques )
X_train_c, X_test_norm_c = train_test_split(X_normal, test_size=0.2, random_state=42)
n   = len(X_test_norm_c)
idx = np.random.choice(len(X_attaque), min(n, len(X_attaque)), replace=False)
X_test_c = np.vstack([X_test_norm_c, X_attaque[idx]])  # normaux + attaques
y_test_c = np.array([0]*n + [1]*len(idx))               # labels correspondants

# construction et entraînement 
ae_c = build_autoencoder(X_train_c.shape[1], latent_dim=16, hidden=[64, 32])
ae_c.fit(X_train_c, X_train_c, epochs=30, batch_size=256, validation_split=0.1, verbose=0)

# mesure du temps et de la mémoire pendant l'inférence
tracemalloc.start()
t0        = time.time()
err_train = np.mean((X_train_c - ae_c.predict(X_train_c, verbose=0))**2, axis=1)
seuil_c   = err_train.mean() + 2 * err_train.std()  # seuil = moyenne + 2 écarts-types
err_test  = np.mean((X_test_c - ae_c.predict(X_test_c, verbose=0))**2, axis=1)
t_ae_c    = round(time.time() - t0, 3)
_, peak   = tracemalloc.get_traced_memory()
tracemalloc.stop()

pred_ae_c = (err_test > seuil_c).astype(int)  # 1 pour anomalie si erreur > seuil
resultats_cicids.append({
    'Dataset': 'CICIDS2017', 'Modèle': 'Autoencoder',
    'Précision'   : round(precision_score(y_test_c, pred_ae_c, zero_division=0), 3),
    'Recall'      : round(recall_score(y_test_c, pred_ae_c, zero_division=0), 3),
    'F1'          : round(f1_score(y_test_c, pred_ae_c, zero_division=0), 3),
    'AUC-ROC'     : round(roc_auc_score(y_test_c, err_test), 3),
    'Temps (s)'   : t_ae_c,
    'Mémoire (MB)': round(peak/1024/1024, 2)
})


#  LSTM et GRU 
SEQ_LEN = 5  # longueur de la fenêtre temporelle

# création des séquences sur les normaux (train) et les attaques (test)
X_seq_c, y_seq_c = creer_sequences(X_normal, SEQ_LEN)
X_tr_seq, X_te_seq, y_tr_seq, y_te_seq = train_test_split(X_seq_c, y_seq_c, test_size=0.2, random_state=42)
X_att_seq_c, y_att_seq_c = creer_sequences(X_attaque[np.random.choice(len(X_attaque), 5000, replace=False)], SEQ_LEN)

# séquences normales et séquences d'attaques
n2 = len(X_te_seq)
X_test_seq_c = np.vstack([X_te_seq[:n2], X_att_seq_c[:n2]])
y_test_seq_c = np.vstack([y_te_seq[:n2], y_att_seq_c[:n2]])
labels_seq_c = np.array([0]*n2 + [1]*n2)

for nom, modele in [('LSTM', build_lstm(SEQ_LEN, X_tr_seq.shape[2])),('GRU',  build_gru(SEQ_LEN,  X_tr_seq.shape[2]))]:
    modele.fit(X_tr_seq, y_tr_seq, epochs=20, batch_size=256,validation_split=0.1, verbose=0)
    # seuil calculé sur les erreurs d'entraînement
    err_tr = np.mean((y_tr_seq - modele.predict(X_tr_seq, verbose=0))**2, axis=1)
    seuil  = err_tr.mean() + 2 * err_tr.std()

    # mesure temps et mémoire pendant l'inférence sur le test
    tracemalloc.start()
    t0 = time.time()
    err_te = np.mean((y_test_seq_c - modele.predict(X_test_seq_c, verbose=0))**2, axis=1)
    t_inf  = round(time.time() - t0, 3)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    pred = (err_te > seuil).astype(int)
    resultats_cicids.append({'Dataset': 'CICIDS2017', 'Modèle': nom,'Précision'   : round(precision_score(labels_seq_c, pred, zero_division=0), 3),'Recall'      : round(recall_score(labels_seq_c, pred, zero_division=0), 3),
        'F1'          : round(f1_score(labels_seq_c, pred, zero_division=0), 3),'AUC-ROC'     : round(roc_auc_score(labels_seq_c, err_te), 3),'Temps (s)'   : t_inf,'Mémoire (MB)': round(peak/1024/1024, 2)})

df_cicids_res = pd.DataFrame(resultats_cicids)
print(df_cicids_res.to_string(index=False))

Évaluation Isolation Forest...
Évaluation LOF...
Évaluation One-Class SVM...
   Dataset           Modèle  Précision  Recall    F1  AUC-ROC  Temps (s)  Mémoire (MB)
CICIDS2017 Isolation Forest      0.459   0.400 0.427    0.338      3.230          4.53
CICIDS2017              LOF      0.533   0.464 0.496    0.428      3.278          1.83
CICIDS2017    One-Class SVM      0.382   0.333 0.356    0.252     45.873          6.17
CICIDS2017      Autoencoder      0.948   0.179 0.301    0.958      5.456         15.23
CICIDS2017             LSTM      0.000   0.000 0.000    0.730      1.814         24.22
CICIDS2017              GRU      0.000   0.000 0.000    0.751      1.649         24.22


## Dataset 2 : UNSW-NB15

In [13]:
# Préparation
X = df_unsw.drop(columns=['Label']).select_dtypes(include=[np.number])
y_unsw = df_unsw['Label'].values
X = X.fillna(0).replace([np.inf, -np.inf], 0)
X_unsw_scaled = StandardScaler().fit_transform(X)

X_u, _, y_u, _ = train_test_split(X_unsw_scaled, y_unsw, train_size=10000, stratify=y_unsw, random_state=42)

print(f"Échantillon UNSW : {X_u.shape} — {y_u.mean():.2%} d'attaques")

Échantillon UNSW : (10000, 44) — 3.17% d'attaques


In [14]:
resultats_unsw = []

# Baselines avec contamination=0.03 car 3% d'attaques dans UNSW
for nom, modele in [('Isolation Forest', IsolationForest(contamination=0.03, random_state=42, n_jobs=-1)),('LOF', LocalOutlierFactor(n_neighbors=5, contamination=0.03)),('One-Class SVM', OneClassSVM(kernel='rbf', nu=0.03, gamma='scale'))]:
    resultats_unsw.append(evaluer_baseline(modele, X_u, y_u, nom, 'UNSW-NB15'))

# Autoencoder avec séparation normaux et attaques
X_normal_u  = X_unsw_scaled[y_unsw == 0][:20000]
X_attaque_u = X_unsw_scaled[y_unsw == 1]
X_train_u, X_test_norm_u = train_test_split(X_normal_u, test_size=0.2, random_state=42)
n   = len(X_test_norm_u)
idx = np.random.choice(len(X_attaque_u), min(n, len(X_attaque_u)), replace=False)
X_test_u = np.vstack([X_test_norm_u, X_attaque_u[idx]])
y_test_u = np.array([0]*n + [1]*len(idx))

ae_u = build_autoencoder(X_train_u.shape[1], latent_dim=8, hidden=[32])
ae_u.fit(X_train_u, X_train_u, epochs=30, batch_size=256, validation_split=0.1, verbose=0)

tracemalloc.start()
t0   = time.time()
err_train = np.mean((X_train_u - ae_u.predict(X_train_u, verbose=0))**2, axis=1)
seuil_u   = err_train.mean() + 2 * err_train.std()
err_test  = np.mean((X_test_u - ae_u.predict(X_test_u, verbose=0))**2, axis=1)
t_ae_u    = round(time.time() - t0, 3)
_, peak   = tracemalloc.get_traced_memory()
tracemalloc.stop()

pred_ae_u = (err_test > seuil_u).astype(int)
resultats_unsw.append({'Dataset': 'UNSW-NB15', 'Modèle': 'Autoencoder','Précision': round(precision_score(y_test_u, pred_ae_u, zero_division=0), 3),'Recall': round(recall_score(y_test_u, pred_ae_u, zero_division=0), 3),'F1': round(f1_score(y_test_u, pred_ae_u, zero_division=0), 3),
    'AUC-ROC' : round(roc_auc_score(y_test_u, err_test), 3),'Temps (s)': t_ae_u, 'Mémoire (MB)': round(peak/1024/1024, 2)})

# LSTM et GRU 
X_seq_u, y_seq_u = creer_sequences(X_normal_u, SEQ_LEN)
X_tr_u, X_te_u, y_tr_u, y_te_u = train_test_split(X_seq_u, y_seq_u, test_size=0.2, random_state=42)
X_att_u, y_att_u = creer_sequences(X_attaque_u[np.random.choice(len(X_attaque_u), 5000, replace=False)], SEQ_LEN)
n2 = len(X_te_u)
X_test_seq_u = np.vstack([X_te_u[:n2], X_att_u[:n2]])
y_test_seq_u = np.vstack([y_te_u[:n2], y_att_u[:n2]])
labels_u  = np.array([0]*n2 + [1]*n2)

for nom, modele in [('LSTM', build_lstm(SEQ_LEN, X_tr_u.shape[2])),('GRU',  build_gru(SEQ_LEN,  X_tr_u.shape[2]))]:
    modele.fit(X_tr_u, y_tr_u, epochs=20, batch_size=256, validation_split=0.1, verbose=0)
    err_tr = np.mean((y_tr_u - modele.predict(X_tr_u, verbose=0))**2, axis=1)
    seuil  = err_tr.mean() + 2 * err_tr.std()

    tracemalloc.start()
    t0  = time.time()
    err_te  = np.mean((y_test_seq_u - modele.predict(X_test_seq_u, verbose=0))**2, axis=1)
    t_inf  = round(time.time() - t0, 3)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    resultats_unsw.append({'Dataset': 'UNSW-NB15', 'Modèle': nom,'Précision': round(precision_score(labels_u, (err_te > seuil).astype(int), zero_division=0), 3),'Recall': round(recall_score(labels_u, (err_te > seuil).astype(int), zero_division=0), 3),
        'F1' : round(f1_score(labels_u, (err_te > seuil).astype(int), zero_division=0), 3),'AUC-ROC'  : round(roc_auc_score(labels_u, err_te), 3),'Temps (s)': t_inf, 'Mémoire (MB)': round(peak/1024/1024, 2)})

df_unsw_res = pd.DataFrame(resultats_unsw)
print(df_unsw_res.to_string(index=False))

  Dataset           Modèle  Précision  Recall    F1  AUC-ROC  Temps (s)  Mémoire (MB)
UNSW-NB15 Isolation Forest      0.330   0.312 0.321    0.964      0.667          3.16
UNSW-NB15              LOF      0.160   0.151 0.156    0.597      0.726          1.83
UNSW-NB15    One-Class SVM      0.252   0.240 0.246    0.867      2.314          3.50
UNSW-NB15      Autoencoder      0.986   0.973 0.979    0.997      5.927          8.75
UNSW-NB15             LSTM      0.915   0.057 0.106    0.961      1.298         13.55
UNSW-NB15              GRU      0.905   0.050 0.095    0.960      1.054         13.54



## Dataset 3 : Cybersecurity Threat Detection Logs

In [15]:
# Encodage
df_enc = df_logs.copy()
for col in ['protocol', 'action', 'log_type']:
    if col in df_enc.columns:
        df_enc[col + '_enc'] = LabelEncoder().fit_transform(df_enc[col].astype(str))

features = ['bytes_transferred'] + [c for c in df_enc.columns if c.endswith('_enc')]
X = df_enc[features].fillna(0)
y_logs = (df_logs['threat_label'] != 'benign').astype(int)
X_logs_scaled = StandardScaler().fit_transform(X)

X_l, _, y_l, _ = train_test_split(X_logs_scaled, y_logs, train_size=10000, stratify=y_logs, random_state=42)

print(f"Échantillon Logs : {X_l.shape} — {y_l.mean():.2%} d'anomalies")

Échantillon Logs : (10000, 4) — 8.04% d'anomalies


In [16]:
resultats_logs = []

# Baselines avec contamination=0.08 car 8% d'anomalies dans Logs
for nom, modele in [('Isolation Forest', IsolationForest(contamination=0.08, random_state=42, n_jobs=-1)),('LOF', LocalOutlierFactor(n_neighbors=5, contamination=0.08)),('One-Class SVM', OneClassSVM(kernel='rbf', nu=0.08, gamma='scale'))]:
    resultats_logs.append(evaluer_baseline(modele, X_l, y_l, nom, 'Logs'))

# Autoencoder avec 50 000 normaux car dataset avec 6M lignes
X_normal_l  = X_logs_scaled[y_logs == 0][:50000]
X_attaque_l = X_logs_scaled[y_logs == 1]
X_train_l, X_test_norm_l = train_test_split(X_normal_l, test_size=0.2, random_state=42)
n  = len(X_test_norm_l)
idx = np.random.choice(len(X_attaque_l), min(n, len(X_attaque_l)), replace=False)
X_test_l = np.vstack([X_test_norm_l, X_attaque_l[idx]])
y_test_l = np.array([0]*n + [1]*len(idx))

# architecture très légère car seulement 4 features
ae_l = build_autoencoder(X_train_l.shape[1], latent_dim=2, hidden=[8])
ae_l.fit(X_train_l, X_train_l, epochs=15, batch_size=512, validation_split=0.1, verbose=0)

tracemalloc.start()
t0   = time.time()
err_train = np.mean((X_train_l - ae_l.predict(X_train_l, verbose=0))**2, axis=1)
seuil_l   = err_train.mean() + 2 * err_train.std()
err_test  = np.mean((X_test_l - ae_l.predict(X_test_l, verbose=0))**2, axis=1)
t_ae_l    = round(time.time() - t0, 3)
_, peak   = tracemalloc.get_traced_memory()
tracemalloc.stop()

pred_ae_l = (err_test > seuil_l).astype(int)
resultats_logs.append({'Dataset': 'Logs', 'Modèle': 'Autoencoder','Précision': round(precision_score(y_test_l, pred_ae_l, zero_division=0), 3),'Recall': round(recall_score(y_test_l, pred_ae_l, zero_division=0), 3),'F1': round(f1_score(y_test_l, pred_ae_l, zero_division=0), 3),
    'AUC-ROC': round(roc_auc_score(y_test_l, err_test), 3),'Temps (s)': t_ae_l, 'Mémoire (MB)': round(peak/1024/1024, 2)})

# LSTM et GRU avec units=32 car seulement 4 features
X_seq_l, y_seq_l = creer_sequences(X_normal_l[:20000], SEQ_LEN)
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_seq_l, y_seq_l, test_size=0.2, random_state=42)
X_att_l, y_att_l = creer_sequences(X_attaque_l[np.random.choice(len(X_attaque_l), 5000, replace=False)], SEQ_LEN)
n2 = len(X_te_l)
X_test_seq_l = np.vstack([X_te_l[:n2], X_att_l[:n2]])
y_test_seq_l = np.vstack([y_te_l[:n2], y_att_l[:n2]])
labels_l  = np.array([0]*n2 + [1]*n2)

for nom, modele in [('LSTM', build_lstm(SEQ_LEN, X_tr_l.shape[2], units=32)),('GRU',  build_gru(SEQ_LEN,  X_tr_l.shape[2], units=32))]:
    modele.fit(X_tr_l, y_tr_l, epochs=15, batch_size=512, validation_split=0.1, verbose=0)
    err_tr = np.nan_to_num(np.mean((y_tr_l - modele.predict(X_tr_l, verbose=0))**2, axis=1))
    seuil  = err_tr.mean() + 2 * err_tr.std()  # nan_to_num évite les erreurs si nan

    tracemalloc.start()
    t0 = time.time()
    err_te  = np.mean((y_test_seq_l - modele.predict(X_test_seq_l, verbose=0))**2, axis=1)
    t_inf   = round(time.time() - t0, 3)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    pred = (err_te > seuil).astype(int)
    resultats_logs.append({'Dataset': 'Logs', 'Modèle': nom,'Précision': round(precision_score(labels_l, pred, zero_division=0), 3),'Recall' : round(recall_score(labels_l, pred, zero_division=0), 3),'F1' : round(f1_score(labels_l, pred, zero_division=0), 3),
        'AUC-ROC' : round(roc_auc_score(labels_l, err_te), 3),'Temps (s)': t_inf, 'Mémoire (MB)': round(peak/1024/1024, 2)})

df_logs_res = pd.DataFrame(resultats_logs)
print(df_logs_res.to_string(index=False))

Dataset           Modèle  Précision  Recall    F1  AUC-ROC  Temps (s)  Mémoire (MB)
   Logs Isolation Forest      0.105   0.104 0.105    0.545      1.133          1.57
   Logs              LOF      0.087   0.087 0.087    0.506      0.121          2.87
   Logs    One-Class SVM      0.080   0.080 0.080    0.456      3.957          0.45
   Logs      Autoencoder      0.469   0.041 0.075    0.494     12.160          3.30
   Logs             LSTM      0.498   0.033 0.061    0.467      2.400          1.33
   Logs              GRU      0.492   0.032 0.061    0.468      1.831          1.33



## Tableau comparatif final

In [17]:
# Fusion de tous les résultats
df_final = pd.concat([df_cicids_res, df_unsw_res, df_logs_res], ignore_index=True)

# Affichage par dataset
for dataset in ['CICIDS2017', 'UNSW-NB15', 'Logs']:
    print(f"Dataset : {dataset}")
    df_d = df_final[df_final['Dataset'] == dataset].drop(columns='Dataset')
    print(df_d.sort_values('AUC-ROC', ascending=False).to_string(index=False))


Dataset : CICIDS2017
          Modèle  Précision  Recall    F1  AUC-ROC  Temps (s)  Mémoire (MB)
     Autoencoder      0.948   0.179 0.301    0.958      5.456         15.23
             GRU      0.000   0.000 0.000    0.751      1.649         24.22
            LSTM      0.000   0.000 0.000    0.730      1.814         24.22
             LOF      0.533   0.464 0.496    0.428      3.278          1.83
Isolation Forest      0.459   0.400 0.427    0.338      3.230          4.53
   One-Class SVM      0.382   0.333 0.356    0.252     45.873          6.17
Dataset : UNSW-NB15
          Modèle  Précision  Recall    F1  AUC-ROC  Temps (s)  Mémoire (MB)
     Autoencoder      0.986   0.973 0.979    0.997      5.927          8.75
Isolation Forest      0.330   0.312 0.321    0.964      0.667          3.16
            LSTM      0.915   0.057 0.106    0.961      1.298         13.55
             GRU      0.905   0.050 0.095    0.960      1.054         13.54
   One-Class SVM      0.252   0.240 0.246    0.

Le tableau comparatif final permet de tirer de nombreuses conclusions sur tous les modèles testés.
Sur le dataset CICIDS2017, l'Autoencoder domine avec une AUC de 0.958 et un f1 de 0.301 ensuite le Le GRU et le LSTM ont des AUC correctes (0.751 et 0.730) mais un f1 nul ainsi ils ne détectent aucune attaque en pratique à cause du seuil inadapté. Le LOF reste bon avec un f1=0.496 mais l'Autoencoder s'impose clairement. Le One-Class SVM est le plus lent (45.87s) sans être le meilleur.
Sur le dataset UNSW-NB15, l'Autoencoder est loin devant les autres avec une AUC de 0.997 et un f1 de 0.979 c'est le meilleur résultat de tous. L'Isolation Forest (AUC=0.964) et le LSTM/GRU (AUC=0.960) sont bons en termes de séparation mais leur recall très faible (entre 5 et 7%) les rend peu utiles en pratique. Le LOF reste le moins performant sur ce dataset.
Sur le dataset Logs, tous les modèles échouent avec des AUC autour de 0.5 soit un niveau aléatoire. C'est la confirmation définitive que les 4 features disponibles ne permettent à aucun algorithme de détecter les anomalies sur ce dataset.
En termes de ressources, le One-Class SVM est systématiquement le plus lent. L'Isolation Forest et le LOF sont les plus légers en mémoire (entre 1 et 5 MB) tandis que l'Autoencoder consomme davantage (entre 8 et 15 MB) mais offre les meilleures performances. Le meilleur compromis performance/ressources est l'Autoencoder sur le dataset UNSW-NB15.